# 00 · Reconocimiento del dataset QQP (PROFECO)

**Objetivo:** decidir si los datos permiten comparar el costo de una canasta entre cadenas, municipios y semanas, y con qué definición.

**Reproducir:** `python -m src.ingest.extract` → `python -m src.ingest.profile` → `python analysis/build_notebook.py`.

Las interpretaciones están en `docs/diccionario_validado.md` y las decisiones en `docs/decisiones.md`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'analysis' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.ingest.profile import PROFILE_DIR  # noqa: E402
from src.ingest.profile_report import alertas  # noqa: E402

pd.set_option('display.max_colwidth', 70)
pd.set_option('display.float_format', '{:,.2f}'.format)
t = {p.stem: pd.read_parquet(p) for p in sorted(PROFILE_DIR.glob('*.parquet'))}
sorted(t)

['archivos',
 'canasta_completitud',
 'canasta_sku_mejor_presencia',
 'cardinalidades',
 'catalogo_por_anio',
 'cobertura_cadenas_basicos',
 'cobertura_estados',
 'coordenadas_resumen',
 'duplicados_multicatalogo',
 'duplicados_por_archivo',
 'estado_variantes',
 'fechas_por_archivo',
 'giro_valores',
 'marca_sin_marca_variantes',
 'nulos_por_columna',
 'precio_resumen',
 'precios_maximos',
 'rechazos',
 'visitas_tienda_quincena']

## 1. Inventario y formato físico

In [2]:
arch = t['archivos'].merge(t['fechas_por_archivo'], on='archivo')
print(f"{len(arch)} CSV · {arch.filas_cargadas.sum():,} filas · "
      f"{arch.bytes.sum() / 1e9:.2f} GB · {arch.min_fecha.min():%Y-%m-%d} a {arch.max_fecha.max():%Y-%m-%d}")
arch.groupby(['codificacion', 'bom', 'formato_fecha']).agg(archivos=('archivo', 'count'), filas=('filas_cargadas', 'sum'), primero=('archivo', 'min'), ultimo=('archivo', 'max'))

58 CSV · 33,581,522 filas · 10.57 GB · 2024-01-02 a 2026-05-29


,,,archivos,filas,primero,ultimo
codificacion,bom,formato_fecha,,,,
cp1252,False,dd/mm/yyyy,2,1007082,QQP_2026/05-2026_Q1.csv,QQP_2026/05-2026_Q2.csv
utf-8-sig,True,yyyy/mm/dd,56,32574440,QQP_2024/01-2024_01.csv,QQP_2026/04-2026_Q2.csv


In [3]:
# Integridad de carga: cada línea física (sin encabezado) debe ser una fila
check = arch.assign(diferencia=arch.lineas_fisicas - 1 - arch.filas_cargadas)
print('archivos con diferencia:', int((check.diferencia != 0).sum()), '· filas rechazadas:', len(t['rechazos']))
check[['archivo', 'filas_cargadas', 'min_fecha', 'max_fecha', 'dias']].tail(6)

archivos con diferencia: 0 · filas rechazadas: 0


,archivo,filas_cargadas,min_fecha,max_fecha,dias
52,QQP_2026/03-2026_Q1.csv,558146,2026-03-02,2026-03-13,11
53,QQP_2026/03-2026_Q2.csv,606522,2026-03-17,2026-03-31,11
54,QQP_2026/04-2026_Q1.csv,480842,2026-04-01,2026-04-15,10
55,QQP_2026/04-2026_Q2.csv,597634,2026-04-16,2026-04-30,11
56,QQP_2026/05-2026_Q1.csv,479993,2026-05-01,2026-05-15,15
57,QQP_2026/05-2026_Q2.csv,527089,2026-05-16,2026-05-29,14


## 2. Alertas automáticas

In [4]:
for a in alertas(t):
    print('•', a)

• **Codificación distinta** en 2 archivo(s): `QQP_2026/05-2026_Q1.csv` (cp1252), `QQP_2026/05-2026_Q2.csv` (cp1252)
• **Formato de fecha distinto** (`dd/mm/yyyy` vs `yyyy/mm/dd`) en: `QQP_2026/05-2026_Q1.csv`, `QQP_2026/05-2026_Q2.csv`
• Filas rechazadas por el lector CSV: **0**
• **Precio extremo**: Prevefem Complex (Caja con 30 Tabletas) a $3,041,791 en Hipermercado Soriana — más de 100× el percentil 99 ($12,599)
• Duplicados exactos: **211** filas; en grano candidato: **636,840** (1.90%), de los cuales 614,653 grupos son la misma observación publicada en varios catálogos y 20,816 tienen precios en conflicto
• **Caracteres perdidos (`?` en lugar de letra acentuada)** en: `QQP_2026/05-2026_Q2.csv` (12,378 filas)
• **Nombres de estado con variantes** (acentos): 7 estados (37 valores originales → 30 normalizados)
• **Marca genérica con variantes de mayúsculas**: `S/m`, `S/M`


## 3. Esquema, nulos y cardinalidad

In [5]:
t['cardinalidades'].merge(t['nulos_por_columna'], on='columna', how='outer')

,columna,distintos,vacios
0,cadena_comercial,264.00,0.00
1,catalogo,12.00,0.00
2,categoria,56.00,0.00
3,coordenadas,"2,228.00",NaN
4,direccion,"3,224.00",0.00
5,estado,37.00,0.00
6,fecha_registro,630.00,0.00
7,giro,18.00,0.00
8,latitud,NaN,791.00
9,longitud,NaN,791.00


## 4. Precio

In [6]:
display(t['precio_resumen'])
t['precios_maximos']

,no_convertible,negativos,ceros,minimo,p01,mediana,p99,maximo
0,0,0,0,1.15,7.00,61.00,"12,599.00","3,041,791.00"


,producto,presentacion,marca,catalogo,precio,cadena_comercial,archivo
0,Prevefem Complex,Caja con 30 Tabletas,S/m,Medicamentos,"3,041,791.00",Hipermercado Soriana,QQP_2024/12-2024_01.csv
1,Pantallas,100 Qned 86 As. 100 Plgs. Qned. Puerto USB. Smart TV.,Lg,Electrodomesticos,"107,691.00",Liverpool,QQP_2026/05-2026_Q2.csv
2,Pantallas,100 Qned 86 As. 100 Plgs. Qned. Puerto USB. Smart TV.,Lg,Electrodomesticos,"107,691.00",Liverpool,QQP_2026/05-2026_Q2.csv
3,Pantallas,100 Qned 86 As. 100 Plgs. Qned. Puerto USB. Smart TV.,Lg,Electrodomesticos,"107,691.00",Liverpool,QQP_2026/05-2026_Q2.csv
4,Lavadoras,Lma 78113 Cbab0 o Cbab00 o Cbab01. 18 Kg. Agitador. Centrifugado. ...,Mabe,Electrodomesticos,"99,999.00",Elektra,QQP_2025/07-2025_01.csv
5,Pantallas,Un 98du9000f. 98 Plgs. Crystal. Puerto USB. Smart TV.,Samsung,Electrodomesticos,"85,713.00",Sears Roebuck de México,QQP_2026/05-2026_Q2.csv
6,Pantallas,Oled 77 C5 Psa. 77 Plgs. Oled. Puerto Usb. Smart Tv.,Lg,Electrodomesticos,"84,999.00",Liverpool,QQP_2026/01-2026_Q2.csv
7,Pantallas,Oled 77 C5 Psa. 77 Plgs. Oled. Puerto Usb. Smart Tv.,Lg,Electrodomesticos,"84,999.00",Sears Roebuck de Mexico,QQP_2026/02-2026_Q2.csv
8,Pantallas,Oled 77 C5 Psa. 77 Plgs. Oled. Puerto Usb. Smart Tv.,Lg,Electrodomesticos,"84,999.00",Sears Roebuck de Mexico,QQP_2026/03-2026_Q1.csv
9,Pantallas,Oled 77 C5 Psa. 77 Plgs. Oled. Puerto Usb. Smart Tv.,Lg,Electrodomesticos,"84,999.00",Liverpool,QQP_2026/03-2026_Q1.csv


El máximo es un error evidente de captura (medicamento de caja con 30 tabletas). Los siguientes son electrodomésticos plausibles: la detección de atípicos debe ser **por producto**, no global.

## 5. Duplicados

In [7]:
dup = t['duplicados_por_archivo']
tot = dup.sum(numeric_only=True)
print(f"duplicados exactos: {tot.dup_exactos:,} · en grano: {tot.dup_grano:,} "
      f"({100 * tot.dup_grano / tot.filas:.2f}%)")
t['duplicados_multicatalogo']

duplicados exactos: 211 · en grano: 636,840 (1.90%)


,combinacion,grupos
0,Basicos + Pacic,340372
1,Frutas y Legumbres + Pacic,265077
2,Electrodomesticos + Juguetes,4295
3,Frutas y Legumbres + Mercados,2593
4,Electrodomesticos + Utiles Escolares,1392
5,Basicos + Especial,862
6,Mercados + Pescados y Mariscos,37
7,Mercados + Pacic,13
8,Basicos + Mercados,12


La mayoría de los duplicados es la misma observación listada en dos catálogos (p. ej. `Basicos + Pacic`). **El catálogo no es parte del grano** (D-006).

## 6. Variantes de texto que partirían series

In [8]:
ev = t['estado_variantes']
display(ev[ev.duplicated('estado_normalizado', keep=False)])
t['marca_sin_marca_variantes']

,estado_normalizado,estado_original,filas,primer_archivo,ultimo_archivo
6,CIUDAD DE MEXICO,Ciudad de Mexico,6137934,QQP_2024/01-2024_01.csv,QQP_2025/11-2025_02.csv
7,CIUDAD DE MEXICO,Ciudad de México,1394570,QQP_2025/12-2025_01.csv,QQP_2026/05-2026_Q2.csv
10,ESTADO DE MEXICO,Estado de Mexico,3656006,QQP_2024/01-2024_01.csv,QQP_2025/11-2025_02.csv
11,ESTADO DE MEXICO,Estado de México,834222,QQP_2025/12-2025_01.csv,QQP_2026/05-2026_Q2.csv
16,MICHOACAN,Michoacan,528782,QQP_2024/01-2024_01.csv,QQP_2025/11-2025_02.csv
17,MICHOACAN,Michoacán,120909,QQP_2025/12-2025_01.csv,QQP_2026/05-2026_Q2.csv
19,NUEVO LEON,Nuevo Leon,844369,QQP_2024/01-2024_01.csv,QQP_2025/11-2025_02.csv
20,NUEVO LEON,Nuevo León,188712,QQP_2025/12-2025_01.csv,QQP_2026/05-2026_Q2.csv
23,QUERETARO,Queretaro,646145,QQP_2024/01-2024_01.csv,QQP_2025/11-2025_02.csv
24,QUERETARO,Querétaro,103111,QQP_2025/12-2025_01.csv,QQP_2026/05-2026_Q2.csv


,marca,anio,filas
0,S/m,2024,4917060
1,S/M,2025,331366
2,S/m,2025,4294517
3,S/M,2026,1867727


## 7. Cobertura

In [9]:
t['catalogo_por_anio'].pivot(index='catalogo', columns='anio', values='filas').fillna(0).astype(int)

anio,2024,2025,2026
catalogo,,,
Basicos,7755715,6888079,2916960
Electrodomesticos,1222507,833885,305403
Especial,0,5189,41178
Frutas y Legumbres,813287,737179,312233
Juguetes,204197,141186,4109
Medicamentos,3342835,3372638,1360813
Mercados,294405,210338,104707
Navideños,35607,29259,9
Pacic,646909,627234,239626


In [10]:
t['cobertura_estados'].sort_values('semanas')

,estado,municipios,semanas,tiendas,filas
12,HIDALGO,2,70,40,105359
22,SINALOA,1,116,27,139180
23,SONORA,1,122,52,504714
4,CHIAPAS,1,122,70,644334
11,GUERRERO,1,123,40,426238
14,MICHOACAN,1,123,58,649691
27,VERACRUZ,3,124,59,737153
20,QUINTANA ROO,2,124,99,990781
2,BAJA CALIFORNIA SUR,1,125,62,700739
3,CAMPECHE,1,125,58,753548


In [11]:
t['cobertura_cadenas_basicos'].head(12)

,cadena,estados,municipios,tiendas,semanas,productos,filas
0,Wal-mart,30,48,64,126,201,3395449
1,Hipermercado Soriana,27,44,54,126,203,2912210
2,Bodega Aurrera,26,45,62,126,199,2306982
3,Chedraui,21,28,29,126,201,1565276
4,Mega Soriana,12,23,24,126,200,1358646
5,Wal-mart Express,8,18,30,126,197,1096980
6,Mercado Soriana,6,11,13,126,191,472643
7,La Comer,4,8,8,126,198,432910
8,Soriana Super,8,9,9,126,194,391059
9,Farmacia Guadalajara,20,23,25,125,154,376424


In [12]:
v = t['visitas_tienda_quincena']
v.assign(pct=100 * v.tienda_quincenas / v.tienda_quincenas.sum())

,dias_con_registro,tienda_quincenas,pct
0,1,1006,9.33
1,2,7465,69.22
2,3,1891,17.54
3,4,298,2.76
4,5,113,1.05
5,6,10,0.09
6,7,1,0.01


Una tienda de las cadenas de referencia se visita típicamente **2 días por quincena**: la semana aislada es un grano demasiado fino para exigir la canasta completa por tienda.

## 8. Viabilidad de la canasta

In [13]:
t['canasta_sku_mejor_presencia']

,producto,presentacion,marca,cadenas,presencia_min_pct,presencia_prom_pct
0,FRIJOL,BOLSA 900 GR. PERUANO,VERDE VALLE,4,3.30,48.90
1,ARROZ,BOLSA 900 GR. SUPER EXTRA. VERDE,SCHETTINO,4,9.40,51.30
2,HUEVO,PAQUETE C/12 BLANCO,SAN JUAN,4,63.00,70.40
3,CARNE RES,1 KG. GRANEL. PANZA O MENUDO. CRUDOS,S/M,4,69.70,75.00
4,HARINA DE MAÍZ,PAQUETE 1 KG.,MASECA,4,72.00,79.80
5,AZÚCAR,BOLSA PLÁSTICO 2 KG. ESTÁNDAR O MORENA,ZULKA,4,73.60,82.30
6,PAPEL HIGIÉNICO,PAQUETE 4 ROLLOS. XXL HOJAS DOBLES,REGIO. LUXURY,4,73.70,77.60
7,ATÚN,BOLSA 78 GR. ALETA AMARILLA. EN TROZOS EN AGUA,DOLORES,4,76.50,81.10
8,CARNE POLLO,1 KG. GRANEL. PIERNA O PIERNA BATE,S/M,4,77.40,81.50
9,PASTA PARA SOPA,PAQUETE 200 GR. SPAGHETTI NO. 5,BARILLA,4,78.00,87.00


In [14]:
t['canasta_completitud'].pivot(index='nivel', columns='cadena', values='pct_completa')

cadena,Bodega Aurrera,Chedraui,Hipermercado Soriana,Wal-mart
nivel,,,,
A. SKU exacto · municipio×quincena,0.00,0.60,19.80,4.00
A. SKU exacto · municipio×semana,0.00,0.00,6.90,1.40
B. Genérico · municipio×mes,79.20,84.30,87.50,85.40
B. Genérico · municipio×quincena,69.60,73.20,79.70,77.60
B. Genérico · municipio×semana,52.10,55.60,65.20,60.70
B. Genérico · municipio×semana (ventana 14 días),68.00,71.60,78.30,76.10


## Conclusiones

1. **El dataset es utilizable**: esquema estable, 0 filas rechazadas, precios y fechas 100% convertibles.
2. **Requiere normalización antes de comparar**: dos formatos de archivo, variantes de estado, marca y presentación, caracteres perdidos en un archivo.
3. **Una canasta de marca fija no es comparable entre las 4 cadenas grandes**: no existe una marca de frijol o arroz presente en todas.
4. **Una canasta genérica con precio por unidad base es viable**, con mejor completitud a nivel quincena o con ventana móvil de 14 días.
5. La cobertura geográfica es de ciudades muestreadas (75 municipios en 30 estados), no de estados completos.